In [1]:
import pandas as pd
import re
import pickle

class FruitData:
    def __init__(self, train_csv, test_csv):
        # Load and combine data
        train = pd.read_csv(train_csv)
        test = pd.read_csv(test_csv)
        self.df = pd.concat([train, test], ignore_index=True)
        self._parse_production()

    def _parse_production(self):
        """Convert 'Annual Production (tonnes)' column into a dictionary column."""
        def parse(prod_str):
            if pd.isna(prod_str) or prod_str.strip() == "":
                return {}
            prod_dict = {}
            parts = prod_str.split('|')
            for part in parts:
                match = re.search(r'([A-Za-z\s]+):\s*([\d,]+)', part)
                if match:
                    fruit = match.group(1).strip()
                    amount_str = match.group(2).replace(',', '')
                    try:
                        amount = int(amount_str)
                        prod_dict[fruit] = amount
                    except:
                        pass
            return prod_dict

        self.df['production_dict'] = self.df['Annual Production (tonnes)'].apply(parse)

    def get_country_info(self, country_name):
        """Print details for a country (case‑insensitive partial match)."""
        mask = self.df['Country'].str.contains(country_name, case=False, na=False)
        results = self.df[mask]

        if len(results) == 0:
            print(f"No country found matching '{country_name}'")
            return None
        elif len(results) > 1:
            print(f"Multiple matches for '{country_name}':")
            for idx, row in results.iterrows():
                print(f"  - {row['Country']}")
            print("\nShowing details for the first match:")
            row = results.iloc[0]
        else:
            row = results.iloc[0]

        # Print details
        print(f"\nCountry: {row['Country']}")
        print(f"Region: {row['Region']}")
        print(f"Main Fruits Produced: {row['Main Fruits Produced']}")
        print(f"Production Rating: {row['Production Rating (1-10)']}")
        print(f"Production Season: {row['Production Season']}")
        print(f"Notes: {row['Notes']}")
        print("\nDetailed production (tonnes):")
        if row['production_dict']:
            for fruit, amount in row['production_dict'].items():
                print(f"  {fruit}: {amount:,}")
        else:
            print("  No detailed production data available.")
        return row

    def get_countries_by_fruit(self, fruit_name):
        """List all countries producing a given fruit (case‑insensitive)."""
        fruit_lower = fruit_name.lower()
        mask = self.df['production_dict'].apply(
            lambda d: any(fruit_lower in k.lower() for k in d.keys())
        )
        results = self.df[mask].copy()

        if len(results) == 0:
            print(f"No country found producing '{fruit_name}'")
            return None

        def get_prod(row):
            for k, v in row['production_dict'].items():
                if fruit_lower in k.lower():
                    return v
            return None

        results['fruit_production'] = results.apply(get_prod, axis=1)
        results = results.sort_values('fruit_production', ascending=False)

        print(f"Countries producing '{fruit_name}':")
        for idx, row in results.iterrows():
            prod = row['fruit_production']
            prod_str = f"{prod:,}" if prod is not None else "N/A"
            print(f"  {row['Country']}: {prod_str} tonnes")
        return results

# -------------------------------------------------------------------
if __name__ == "__main__":
    # Make sure the CSV files are in the same folder as this script
    fruit_data = FruitData('africa_fruit_train.csv', 'africa_fruit_test.csv')

    with open('fruit_data.pkl', 'wb') as f:
        pickle.dump(fruit_data, f)

    print("✅ fruit_data.pkl has been created successfully!")
!pip install pandas

import pickle

# Load the pickled object (the class must be defined in this script or imported)
# If you run this in a separate file, copy the FruitData class definition above.
# Alternatively, you can import it from a module.

with open('fruit_data.pkl', 'rb') as f:
    fruit_data = pickle.load(f)

# Now query any country
fruit_data.get_country_info("Nigeria")
fruit_data.get_country_info("South Africa")
fruit_data.get_country_info("Ghana")

✅ fruit_data.pkl has been created successfully!

Country: Nigeria
Region: West Africa
Main Fruits Produced: Mangoes, Bananas, Pineapple, Papaya, Watermelon, Citrus
Production Rating: 9
Production Season: Year-round (varies by crop and region)
Notes: Africa's most populous nation; massive domestic production; export potential underutilized

Detailed production (tonnes):
  Mangoes: 900,000
  Pineapple: 900,000
  Bananas: 2,800,000

Country: South Africa
Region: Southern Africa
Main Fruits Produced: Oranges, Grapes, Apples, Mangoes, Avocados, Pears
Production Rating: 10
Production Season: Year-round (Citrus: May-Oct; Grapes: Jan-Apr; Apples: Feb-Apr)
Notes: Africa's top fruit exporter; world-class wine and table grape industry

Detailed production (tonnes):
  Citrus: 2,700,000
  Grapes: 1,900,000
  Apples: 950,000
  Avocados: 120,000

Country: Ghana
Region: West Africa
Main Fruits Produced: Pineapple, Mangoes, Bananas, Watermelon, Papaya
Production Rating: 8
Production Season: Year-round 


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Country                                                                   Ghana
Region                                                              West Africa
Main Fruits Produced            Pineapple, Mangoes, Bananas, Watermelon, Papaya
Annual Production (tonnes)                Pineapple: 750,000 | Mangoes: 150,000
Production Rating (1-10)                                                      8
Production Season             Year-round (Pineapple: Mar-Jul; Mangoes: Mar-Jun)
Notes                         Major pineapple exporter to Europe; growing ma...
production_dict                        {'Pineapple': 750000, 'Mangoes': 150000}
Name: 35, dtype: object